# ABSA Model Sweep With Google Drive Outputs

Run one section at a time. All outputs are saved to Google Drive under `MyDrive/NLP_outputs/sweeps`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = "https://github.com/ngocvo2511/NLP.git"
PROJECT_DIR = "/content/NLP"
OUTPUT_ROOT = "/content/drive/MyDrive/NLP_outputs/sweeps"
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.environ["OUTPUT_ROOT"] = OUTPUT_ROOT

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}
!pip install -q transformers accelerate datasets seqeval scikit-learn sentencepiece protobuf
print("Outputs:", OUTPUT_ROOT)

## XLM-R Base: BIO With Exact-Span Checkpoint Selection

This reruns the strongest current family, but now selects the best checkpoint by exact span F1 instead of token F1.

In [ ]:
!python -m src.absa.train_transformer \
  --model-name FacebookAI/xlm-roberta-base \
  --data-dir data \
  --output-dir "$OUTPUT_ROOT/xlmr-base-bio-exact" \
  --tag-scheme bio \
  --class-weight none \
  --metric-for-best-model exact_span_f1 \
  --epochs 6 \
  --batch-size 8 \
  --learning-rate 2e-5 \
  --warmup-ratio 0.06 \
  --save-total-limit 2 \
  --fp16

!python -m src.absa.predict_transformer --model-dir "$OUTPUT_ROOT/xlmr-base-bio-exact" --input data/dev.jsonl --output "$OUTPUT_ROOT/xlmr_base_bio_exact_dev_predictions.jsonl"
!python -m src.absa.evaluate --gold data/dev.jsonl --pred "$OUTPUT_ROOT/xlmr_base_bio_exact_dev_predictions.jsonl" --json-output "$OUTPUT_ROOT/xlmr_base_bio_exact_dev_metrics.json"

## mDeBERTa-v3 Base

A multilingual DeBERTa variant. It is often competitive with XLM-R on multilingual NLU.

In [ ]:
!python -m src.absa.train_transformer \
  --model-name microsoft/mdeberta-v3-base \
  --data-dir data \
  --output-dir "$OUTPUT_ROOT/mdeberta-v3-base-bio" \
  --tag-scheme bio \
  --class-weight none \
  --metric-for-best-model exact_span_f1 \
  --epochs 6 \
  --batch-size 8 \
  --learning-rate 2e-5 \
  --warmup-ratio 0.06 \
  --save-total-limit 2 \
  --fp16

!python -m src.absa.predict_transformer --model-dir "$OUTPUT_ROOT/mdeberta-v3-base-bio" --input data/dev.jsonl --output "$OUTPUT_ROOT/mdeberta_v3_base_bio_dev_predictions.jsonl"
!python -m src.absa.evaluate --gold data/dev.jsonl --pred "$OUTPUT_ROOT/mdeberta_v3_base_bio_dev_predictions.jsonl" --json-output "$OUTPUT_ROOT/mdeberta_v3_base_bio_dev_metrics.json"

## Multilingual BERT Cased

Smaller and older than XLM-R, but useful as a simple multilingual baseline.

In [ ]:
!python -m src.absa.train_transformer \
  --model-name google-bert/bert-base-multilingual-cased \
  --data-dir data \
  --output-dir "$OUTPUT_ROOT/mbert-cased-bio" \
  --tag-scheme bio \
  --class-weight none \
  --metric-for-best-model exact_span_f1 \
  --epochs 6 \
  --batch-size 8 \
  --learning-rate 2e-5 \
  --warmup-ratio 0.06 \
  --save-total-limit 2 \
  --fp16

!python -m src.absa.predict_transformer --model-dir "$OUTPUT_ROOT/mbert-cased-bio" --input data/dev.jsonl --output "$OUTPUT_ROOT/mbert_cased_bio_dev_predictions.jsonl"
!python -m src.absa.evaluate --gold data/dev.jsonl --pred "$OUTPUT_ROOT/mbert_cased_bio_dev_predictions.jsonl" --json-output "$OUTPUT_ROOT/mbert_cased_bio_dev_metrics.json"

## XLM-R Large

Run this only if GPU memory allows it. On T4, use small batch size plus gradient accumulation.

In [ ]:
!python -m src.absa.train_transformer \
  --model-name FacebookAI/xlm-roberta-large \
  --data-dir data \
  --output-dir "$OUTPUT_ROOT/xlmr-large-bio" \
  --tag-scheme bio \
  --class-weight none \
  --metric-for-best-model exact_span_f1 \
  --epochs 4 \
  --batch-size 2 \
  --gradient-accumulation-steps 4 \
  --learning-rate 1e-5 \
  --warmup-ratio 0.06 \
  --save-total-limit 1 \
  --fp16

!python -m src.absa.predict_transformer --model-dir "$OUTPUT_ROOT/xlmr-large-bio" --input data/dev.jsonl --output "$OUTPUT_ROOT/xlmr_large_bio_dev_predictions.jsonl" --batch-size 8
!python -m src.absa.evaluate --gold data/dev.jsonl --pred "$OUTPUT_ROOT/xlmr_large_bio_dev_predictions.jsonl" --json-output "$OUTPUT_ROOT/xlmr_large_bio_dev_metrics.json"

## PhoBERT Base v2

PhoBERT uses a slow tokenizer in standard Transformers and its model card recommends word-segmented input. This experiment uses `--tokenizer-alignment word` to keep original character offsets while expanding labels to PhoBERT subwords. It is a clean offset-preserving PhoBERT trial, but not yet a full VnCoreNLP word-segmented PhoBERT setup.

In [ ]:
!python -m src.absa.train_transformer \
  --model-name vinai/phobert-base-v2 \
  --data-dir data \
  --output-dir "$OUTPUT_ROOT/phobert-base-v2-bio-wordalign" \
  --tag-scheme bio \
  --class-weight none \
  --metric-for-best-model exact_span_f1 \
  --tokenizer-alignment word \
  --epochs 6 \
  --batch-size 8 \
  --learning-rate 2e-5 \
  --warmup-ratio 0.06 \
  --save-total-limit 2

!python -m src.absa.predict_transformer --model-dir "$OUTPUT_ROOT/phobert-base-v2-bio-wordalign" --input data/dev.jsonl --output "$OUTPUT_ROOT/phobert_base_v2_bio_wordalign_dev_predictions.jsonl"
!python -m src.absa.evaluate --gold data/dev.jsonl --pred "$OUTPUT_ROOT/phobert_base_v2_bio_wordalign_dev_predictions.jsonl" --json-output "$OUTPUT_ROOT/phobert_base_v2_bio_wordalign_dev_metrics.json"